# `get_final_feature_sets_and_coefficients`

## Purpose

Obtain feature sets and respective coefficients for models with best performance.

## Previous notebook

`train_non_svc_models`/`train_svc_models`

## Next notebook

~

## Output files



## Files required

- `base_cohort_with_labs_and_vitals.parquet` (in directory specified by `AKI_DATA_PDC` environment variable)

# Imports

In [1]:
from set_env_vars import set_all_env_vars
set_all_env_vars()

import numpy as np
import pandas as pd

import os
import pickle
# import datetime
# import time

# import random

import matplotlib.pyplot as plt
import seaborn as sns

# from scipy.stats import wilcoxon

from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.naive_bayes import GaussianNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier

# from sklearn.model_selection import train_test_split, cross_val_score, cross_validate, StratifiedKFold
# from sklearn.metrics import confusion_matrix, precision_score, recall_score, roc_curve, roc_auc_score, average_precision_score, precision_recall_curve, RocCurveDisplay, accuracy_score

import prepare_data

original_dir = os.getcwd()
os.chdir('../jupyter')
# import load_data
# import feature_name_functions
import cutpoint_analysis
os.chdir(original_dir)

# Load data

In [2]:
train_cohort_ll = prepare_data.load_and_process_cohort('train', 'latest')
train_cohort_med = prepare_data.load_and_process_cohort('train', 'median')

/mnt/batch/tasks/shared/LS_root/mounts/clusters/sdrury-compute/code/Users/Stephen_Drury/vps-peds-aki/azure_ml/prepare_data.py:63: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  cohort[feature].fillna(cohort[feature+'_for_imputation'], inplace=True)
/mnt/batch/tasks/shared/LS_root/mounts/clusters/sdrury-compute/code/Users/Stephen_Drury/vps-peds-aki/azure_ml/prepare_data.py:110: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  cohort[feature] = [(x - min_val) / (max_val - min_val) for x in cohort[feature]]


# Load feature sets

## Define function to get union and intersection of trimmed feature sets

In [3]:
def get_feature_set_union_and_intersect(feature_set_list, return_as_dict=False, verbosity=0):
    union_of_sets = []

    for i, fs in enumerate(feature_set_list):
        if verbosity > 0:
            print('Set %d has length %d' % (i+1, len(fs)))
        for feature in fs:
            if feature not in union_of_sets:
                union_of_sets.append(feature)
    if verbosity > 0:
        print('Union set has length %d' % len(union_of_sets))
    
    intersection_of_sets = [
        feature for feature in feature_set_list[0] if (
            all([feature in feature_set_list[i] for i in range(len(feature_set_list))])
        )
    ]
    if verbosity > 0:
        print('Intersection set has length %d' % len(intersection_of_sets))
    
    if return_as_dict:
        feature_set_dict = {
            'union': union_of_sets,
            'intersection': intersection_of_sets
        }
    
    return union_of_sets, intersection_of_sets

## Get dictionaries of feature sets

In [4]:
ll_feature_set_dict = dict()
med_feature_set_dict = dict()

# Latest lab imputation

## Logistic regression only

### 3 iter

with open('pickle/trimmed_feature_sets/ll_auroc_logreg_only_3_iter.pickle', 'rb') as infile:
    union, intersect = get_feature_set_union_and_intersect(pickle.load(infile))
    ll_feature_set_dict['ll_roc_logreg_only_intersect'] = intersect
    ll_feature_set_dict['ll_roc_logreg_only_union'] = union

with open('pickle/trimmed_feature_sets/ll_auprc_logreg_only_3_iter.pickle', 'rb') as infile:
    union, intersect = get_feature_set_union_and_intersect(pickle.load(infile))
    ll_feature_set_dict['ll_prc_logreg_only_intersect'] = intersect
    ll_feature_set_dict['ll_prc_logreg_only_union'] = union
    
### 10 iter

with open('pickle/trimmed_feature_sets/ll_auroc_logreg_only_10_iter.pickle', 'rb') as infile:
    union, intersect = get_feature_set_union_and_intersect(pickle.load(infile))
    ll_feature_set_dict['ll_roc_logreg_only_10_iter_intersect'] = intersect
    ll_feature_set_dict['ll_roc_logreg_only_10_iter_union'] = union

with open('pickle/trimmed_feature_sets/ll_auprc_logreg_only_10_iter.pickle', 'rb') as infile:
    union, intersect = get_feature_set_union_and_intersect(pickle.load(infile))
    ll_feature_set_dict['ll_prc_logreg_only_10_iter_intersect'] = intersect
    ll_feature_set_dict['ll_prc_logreg_only_10_iter_union'] = union
    
## SVC and logistic regression

with open('pickle/trimmed_feature_sets/ll_auroc_svc_3_iter.pickle', 'rb') as infile:
    union, intersect = get_feature_set_union_and_intersect(pickle.load(infile))
    ll_feature_set_dict['ll_roc_svc_intersect'] = intersect
    ll_feature_set_dict['ll_roc_svc_union'] = union

with open('pickle/trimmed_feature_sets/ll_auprc_svc_3_iter.pickle', 'rb') as infile:
    union, intersect = get_feature_set_union_and_intersect(pickle.load(infile))
    ll_feature_set_dict['ll_prc_svc_intersect'] = intersect
    ll_feature_set_dict['ll_prc_svc_union'] = union
    
    
    
# Median imputation

## Logistic regression only

### 3 iter
    
with open('pickle/trimmed_feature_sets/med_auroc_logreg_only_3_iter.pickle', 'rb') as infile:
    union, intersect = get_feature_set_union_and_intersect(pickle.load(infile))
    med_feature_set_dict['med_roc_logreg_only_intersect'] = intersect
    med_feature_set_dict['med_roc_logreg_only_union'] = union

with open('pickle/trimmed_feature_sets/med_auprc_logreg_only_3_iter.pickle', 'rb') as infile:
    union, intersect = get_feature_set_union_and_intersect(pickle.load(infile))
    med_feature_set_dict['med_prc_logreg_only_intersect'] = intersect
    med_feature_set_dict['med_prc_logreg_only_union'] = union
    
### 10 iter

with open('pickle/trimmed_feature_sets/med_auroc_logreg_only_10_iter.pickle', 'rb') as infile:
    union, intersect = get_feature_set_union_and_intersect(pickle.load(infile))
    med_feature_set_dict['med_roc_logreg_only_10_iter_intersect'] = intersect
    med_feature_set_dict['med_roc_logreg_only_10_iter_union'] = union

with open('pickle/trimmed_feature_sets/med_auprc_logreg_only_10_iter.pickle', 'rb') as infile:
    union, intersect = get_feature_set_union_and_intersect(pickle.load(infile))
    med_feature_set_dict['med_prc_logreg_only_10_iter_intersect'] = intersect
    med_feature_set_dict['med_prc_logreg_only_10_iter_union'] = union
    
## SVC and logistic regression

with open('pickle/trimmed_feature_sets/med_auroc_svc_3_iter.pickle', 'rb') as infile:
    union, intersect = get_feature_set_union_and_intersect(pickle.load(infile))
    med_feature_set_dict['med_roc_svc_intersect'] = intersect
    med_feature_set_dict['med_roc_svc_union'] = union

with open('pickle/trimmed_feature_sets/med_auprc_svc_3_iter.pickle', 'rb') as infile:
    union, intersect = get_feature_set_union_and_intersect(pickle.load(infile))
    med_feature_set_dict['med_prc_svc_intersect'] = intersect
    med_feature_set_dict['med_prc_svc_union'] = union

## Create dictionaries of feature sets with aggregate info removed

In [5]:
ll_feature_set_dict_no_agg = dict()
med_feature_set_dict_no_agg = dict()

all_features_no_agg = []

for key in ll_feature_set_dict.keys():
    ll_feature_set_dict_no_agg[key] = []
    for feature in ll_feature_set_dict[key]:
        feature_no_agg = feature.replace('_min', '').replace('_max', '').replace('_mean', '').replace('_median', '')
        if feature_no_agg not in ll_feature_set_dict_no_agg[key]:
            ll_feature_set_dict_no_agg[key].append(feature_no_agg)
        if feature_no_agg not in all_features_no_agg:
            all_features_no_agg.append(feature_no_agg)
            
for key in med_feature_set_dict.keys():
    med_feature_set_dict_no_agg[key] = []
    for feature in med_feature_set_dict[key]:
        feature_no_agg = feature.replace('_min', '').replace('_max', '').replace('_mean', '').replace('_median', '')
        if feature_no_agg not in med_feature_set_dict_no_agg[key]:
            med_feature_set_dict_no_agg[key].append(feature_no_agg)
        if feature_no_agg not in all_features_no_agg:
            all_features_no_agg.append(feature_no_agg)

In [6]:
print('\n'.join(all_features_no_agg))

LACTATE
CREATININE
POTASSIUM
INR
temp
resp_rate
weight
CRP
fio2
RDW
sbp
GLUCOSE
pox
MAGNESIUM
MCV
NEUTRO_PCT
BUN
PROCALCITONIN
FIBRINOGEN
ALC
AST
LDH
ALBUMIN
PH
map
CALCIUM_ION
PO2
PLTS
BASE_DEF
ALT
BASE_EXC
BICARBONATE
FERRITIN
D_DIMER
CHLORIDE
CALCIUM_TOT
ALP
ageatadmission
dbp
ESR
pulse
ANC
WBC


# Define a function to get coefficients for logistic regression model by feature selection parameters

In [7]:
print('\n'.join(ll_feature_set_dict.keys()))

ll_roc_logreg_only_intersect
ll_roc_logreg_only_union
ll_prc_logreg_only_intersect
ll_prc_logreg_only_union
ll_roc_logreg_only_10_iter_intersect
ll_roc_logreg_only_10_iter_union
ll_prc_logreg_only_10_iter_intersect
ll_prc_logreg_only_10_iter_union
ll_roc_svc_intersect
ll_roc_svc_union
ll_prc_svc_intersect
ll_prc_svc_union


In [24]:
def get_logreg_coefficients(
    ll_dict,
    med_dict,
    train_cohort_ll,
    train_cohort_med,
    imputation_method, # 'll' or 'med'
    fs_metric, # 'roc' or 'prc'
    fs_model_type, # 'logreg_only' or 'svc'
    trim_iter_count, # 3 or 10
    fs_combination, # 'union' or 'intersect'
    verbosity=0
):
    # Get feature set key from selection parameters
    fs_key = '_'.join([
        imputation_method,
        fs_metric,
        fs_model_type
    ] +  (['10_iter'] if trim_iter_count==10 else []) + [
        fs_combination
    ]
    )
    
    # Get feature set from dict using key, then get training data (X, y)
    if imputation_method == 'll':
        feature_set = ll_dict[fs_key]
        X = train_cohort_ll[feature_set]
        y = train_cohort_ll['aki_72hrs_any']
    else:
        feature_set = med_dict[fs_key]
        X = train_cohort_med[feature_set]
        y = train_cohort_med['aki_72hrs_any']
        
    if verbosity > 0:
        print('Feature count: %d' % len(feature_set))
        
    # Train model
    model = LogisticRegression(class_weight='balanced', max_iter=1e7, random_state=343)
    model.fit(X, y)
    
    # Get coefficients for each feature
#     coef_dict = dict()
    
#     for feature, coef in zip(X.columns, model.coef_):
#         coef_dict[feature] = coef
        
#     coef_dict['intercept'] = model.intercept_

    coef_df = pd.DataFrame(
        zip(
            list(X.columns) + ['[intercept]'],
            [coef[0] for coef in np.transpose(model.coef_)]+[model.intercept_[0]]
        ),
        columns=['feature', 'coefficient']
    )
    
    return coef_df

# Get features and coefficients for feature set that resulted in best logistic regression model

## Load performance results of models

In [20]:
results_df = pd.read_csv('all_model_performance_results.csv')

In [21]:
results_df.head()

,imputation,fs_metric,logreg_only,trimming_iter_count,fs_combination,model_type,accuracy_cp50,tpr_cp50,tnr_cp50,accuracy_cp90,tpr_cp90,tnr_cp90,auroc,auprc,PPV_cp50,PPV_cp90,feature_set_length,fs_label_full
0,ll,prc,1,10,intersect,DecisionTree,0.951384,0.237099,0.970773,0.951384,0.237099,0.970773,0.604757,0.073266,0.180467,0.180467,42,ll_prc_logreg_only_10_iter_
1,ll,prc,1,10,intersect,Gradient Boosting Classifier,0.523903,0.955370,0.512191,0.913015,0.747559,0.917506,0.915431,0.389509,0.050479,0.197422,42,ll_prc_logreg_only_10_iter_
2,ll,prc,1,10,intersect,Logistic Regression,0.519332,0.945607,0.507761,0.904980,0.592748,0.913455,0.874927,0.172813,0.049561,0.156769,42,ll_prc_logreg_only_10_iter_
3,ll,prc,1,10,intersect,Naive Bayes Gaussian,0.522944,0.935844,0.511736,0.888725,0.287308,0.905050,0.820622,0.079169,0.049455,0.075903,42,ll_prc_logreg_only_10_iter_
4,ll,prc,1,10,intersect,Random Forest,0.550293,0.953975,0.539335,0.912941,0.746165,0.917468,0.912198,0.359365,0.053221,0.197053,42,ll_prc_logreg_only_10_iter_


In [22]:
results_df[
    results_df['model_type']=='Logistic Regression'
].sort_values('PPV_cp90', ascending=False).head(1).T

,28
imputation,ll
fs_metric,prc
logreg_only,0
trimming_iter_count,3
fs_combination,intersect
model_type,Logistic Regression
accuracy_cp50,0.519258
tpr_cp50,0.863319
tnr_cp50,0.509919
accuracy_cp90,0.909476


In [29]:
best_logreg_model_coef_df = get_logreg_coefficients(
    ll_feature_set_dict,
    med_feature_set_dict,
    train_cohort_ll,
    train_cohort_med,
    'll', # 'll' or 'med'
    'prc', # 'roc' or 'prc'
    'logreg_only', # 'logreg_only' or 'svc'
    3, # 3 or 10
    'intersect', # 'union' or 'intersect',
    verbosity=1
)

best_logreg_model_coef_df.to_csv('logreg_model_coefficients/ll_prc_logreg_only_3_iter_intersection.csv', index=False)

Feature count: 23


In [9]:
# for imputation_method in ['ll', 'med']: # 'll' or 'med'
#     for fs_metric in ['roc', 'prc']: # 'roc' or 'prc'
#         for fs_model_type in ['logreg_only', 'svc']: # True or False
#             for trim_iter_count in [3, 10]: # 3 or 10
#                 for fs_combination in ['union', 'intersect']: # 'union' or 'intersect'
#                     if (fs_model_type=='svc' and trim_iter_count==10) == False:
#                         test_key = get_logreg_coefficients(
#                             ll_feature_set_dict,
#                             med_feature_set_dict,
#                             train_cohort_ll,
#                             train_cohort_med,
#                             imputation_method, # 'll' or 'med'
#                             fs_metric, # 'roc' or 'prc'
#                             fs_model_type, # True or False
#                             trim_iter_count, # 3 or 10
#                             fs_combination # 'union' or 'intersect'
#                         )
#                         if imputation_method == 'll':
#                             if test_key not in ll_feature_set_dict.keys():
#                                 print(test_key)
#                         else:
#                             if test_key not in med_feature_set_dict.keys():
#                                 print(test_key)

None
None
None
None
None
None
None



KeyboardInterrupt

